# 🔬 Harmonic Math Sandbox (Tempo Class)
Testing the $O(1)$ logarithmic modulo space tracking in a simplified environment without complex jailbreak triggers.

In [11]:
import librosa
import numpy as np
import torchaudio
import torch
import time
from collections import deque
import warnings
warnings.filterwarnings('ignore')

import sys
import os
sys.path.append(os.path.abspath('../..'))
import core.Listener as ListenerModule
from IPython.display import display, clear_output

def default_infos():
    return {
        "startServer"     : False ,
        "useMicrophone"   : True  ,
        "HARDWARE_MODE"   : "simulation",
        "onRaspberry"     : False  ,
        "printTimeOfCalculation" : False ,
        "printModesDetails"      : True ,
        "printMicrophoneDetails" : False ,
        "printAppDetails"        : False ,
        "printAsservmentDetails" : False ,
        "printConfigurationLoads": False ,
        "printConfigChanges"     : False ,
        "modesToPrintDetails"    : ["PSG"]
    }



In [ ]:
root = '../../assets/musics/mp3_files/'
song_files = [
    'Palladium',
    'Pumped Up Kicks',
    'Nobody Rules the Streets',
]
real_bpms = [145, 128, 128]

for song_id in range(len(song_files)):
    song_files[song_id] = root + song_files[song_id] + ".mp3"

y_list = []
onset_list = []
librosa_dir = os.path.join(root, 'librosa')
os.makedirs(librosa_dir, exist_ok=True)

for i, f in enumerate(song_files):
    basename = os.path.basename(f)
    save_path = os.path.join(librosa_dir, f"{basename}.npz")
    
    if os.path.exists(save_path):
        data = np.load(save_path, allow_pickle=True)
        y = data['y']
        sr = int(data['sr'])
        onset = data['onset']
    else:
        y, sr = librosa.load(f, sr=44100)
        onset = librosa.onset.onset_strength(y=y, sr=sr)
        np.savez(save_path, y=y, sr=sr, onset=onset)
        
    y_list.append(y)
    onset_list.append(onset)



FileNotFoundError: [Errno 2] No such file or directory: '../assets/musics/mp3_files/Palladium.mp3'

In [ ]:
class Robust_Simulated_Microphone:
    def __init__(self, y_full_array, bandValues, infos):
        self.bandValues = bandValues
        self.nb_of_fft_band = len(self.bandValues)
        self.sample_rate = 44100
        self.buffer_size = 1024 
        self.audio_data = np.zeros(self.buffer_size)
        self.full_audio = y_full_array
        self.total_samples = len(self.full_audio)
        self.current_pos = 0
        fft_size = self.buffer_size // 2 + 1
        self.weight_matrix = np.zeros((self.nb_of_fft_band, fft_size))
        
        def hz_to_mel(f): return 2595 * np.log10(1 + f / 700.0)
        def mel_to_hz(m): return 700 * (10**(m / 2595.0) - 1)
        
        lower_mel = hz_to_mel(20)
        upper_mel = hz_to_mel(20000)
        mel_points = np.linspace(lower_mel, upper_mel, self.nb_of_fft_band + 2)
        hz_points = mel_to_hz(mel_points)
        bin_points = np.floor((self.buffer_size + 1) * hz_points / self.sample_rate).astype(int)
        
        for i in range(self.nb_of_fft_band):
            start = min(bin_points[i], fft_size - 1)
            mid = min(bin_points[i + 1], fft_size - 1)
            end = min(bin_points[i + 2], fft_size - 1)
            if mid > start:
                self.weight_matrix[i, start:mid] = np.linspace(0, 1, mid - start, endpoint=False)
            if end > mid:
                self.weight_matrix[i, mid:end] = np.linspace(1, 0, end - mid, endpoint=False)
            band_sum = np.sum(self.weight_matrix[i, :])
            if band_sum > 0:
                self.weight_matrix[i, :] /= band_sum
                
        self.raw_fft_history = None

    def pop_chunk(self, chunk_size=1024):
        if self.current_pos + chunk_size > self.total_samples:
            return False 
        incoming = self.full_audio[self.current_pos : self.current_pos + chunk_size]
        self.current_pos += chunk_size
        self.audio_data = np.roll(self.audio_data, -chunk_size)
        self.audio_data[-chunk_size:] = incoming
        return True

    def calculate_fft(self):
        windowed_data = self.audio_data * np.hanning(self.buffer_size)
        fft_result = np.abs(np.fft.rfft(windowed_data))
        scale = 150.0 / (self.buffer_size / 1024.0)
        mel_bands = np.dot(self.weight_matrix, fft_result) * scale
        for i in range(self.nb_of_fft_band):
            self.bandValues[i] = int(mel_bands[i])
        self.raw_fft_history = fft_result



In [ ]:
# THE HARMONIC MATH CORE (SIMPLIFIED)
def bpm_to_class(bpm):
    '''Map BPM to a float in [0, 1) based on octave'''
    return np.log2(bpm / 60.0) % 1.0

def class_to_bpm_candidates(bpm_class):
    '''Returns the most common harmonic multipliers for a given class'''
    base_bpm = 60.0 * (2 ** bpm_class)
    return [
        base_bpm * 0.5,    # e.g., 50
        base_bpm * 0.75,   # e.g., 75
        base_bpm * 1.0,    # e.g., 100
        base_bpm * 1.5,    # e.g., 150
        base_bpm * 2.0     # e.g., 200
    ]

def tempo_class_distance(f1, f2):
    '''Shortest circular distance on [0, 1)'''
    d = abs(f1 - f2)
    return min(d, 1.0 - d)

def harmonic_alignment(current_class, long_term_class):
    '''Checks straight octaves AND perfect fifths (1.5x) to safely align and find distance'''
    shift = np.log2(1.5) # approx 0.58496
    d_oct = tempo_class_distance(current_class, long_term_class)
    d_fifth_up = tempo_class_distance(current_class, (long_term_class + shift) % 1.0)
    d_fifth_down = tempo_class_distance(current_class, (long_term_class - shift) % 1.0)
    
    min_d = min(d_oct, d_fifth_up, d_fifth_down)
    
    if min_d == d_oct:
        aligned_class = current_class
    elif min_d == d_fifth_up:
        aligned_class = (current_class - shift) % 1.0
    else:
        aligned_class = (current_class + shift) % 1.0
        
    return min_d, aligned_class



In [ ]:
# THE CANDIDATE EVALUATOR (Options A, B, C)
def evaluate_specific_bpms(odf_buffer, candidate_bpms, expected_phase=None, tau_power=1.0):
    odf_size = len(odf_buffer)
    decay_curve = np.exp(-1.5 * np.linspace(1.0, 0.0, odf_size))
    weighted_buffer = odf_buffer * decay_curve
    
    # Pre-compute zero-mean buffer for Pearson
    buffer_mean = np.mean(weighted_buffer)
    buffer_centered = weighted_buffer - buffer_mean
    buffer_std = np.sqrt(np.sum(buffer_centered**2)) + 1e-6
    
    btrack_fps = 60.0
    buffer_indices = np.arange(odf_size)
    const_part = buffer_indices - (odf_size - 1)
    
    best_score_pearson = -float('inf')
    best_bpm_pearson = candidate_bpms[0]
    
    best_score_ratio = -float('inf')
    best_bpm_ratio = candidate_bpms[0]
    
    best_score_density = -float('inf')
    best_bpm_density = candidate_bpms[0]
    
    for bpm_val in candidate_bpms:
        if not (40.0 <= bpm_val <= 240.0):
            continue
            
        tau_val = 60.0 * btrack_fps / bpm_val
        p_max = int(np.ceil(tau_val))
        
        p_arr = np.arange(p_max)[:, None]
        phase_float = (const_part[None, :] + p_arr) % tau_val
        norm_phi = phase_float / tau_val 
        
        # --- SHARP TRIANGLE PULSE TEMPLATE ---
        beat_dist = np.minimum(norm_phi, 1.0 - norm_phi)
        template_vals = np.full((p_max, odf_size), -0.5)
        mask_beat = beat_dist < 0.1
        template_vals[mask_beat] = 1.0 - (beat_dist[mask_beat] / 0.1)
        
        # --- OPTION A: Pearson Correlation (The Math Standard) ---
        template_mean = np.mean(template_vals, axis=1, keepdims=True)
        template_centered = template_vals - template_mean
        template_std = np.sqrt(np.sum(template_centered**2, axis=1)) + 1e-6
        
        p_scores_pearson = np.sum(buffer_centered[None, :] * template_centered, axis=1) / (buffer_std * template_std)
        
        if np.max(p_scores_pearson) > best_score_pearson:
            best_score_pearson = np.max(p_scores_pearson)
            best_bpm_pearson = bpm_val
            
        # --- OPTION B: Peak vs Valley Ratio (The Sub-Harmonic Killer) ---
        template_peaks = (beat_dist < 0.1).astype(float)
        template_valleys = (beat_dist >= 0.4).astype(float)
        
        peak_energy = np.sum(weighted_buffer[None, :] * template_peaks, axis=1)
        valley_energy = np.sum(weighted_buffer[None, :] * template_valleys, axis=1)
        
        p_scores_ratio = peak_energy / (valley_energy + 1.0)
        
        if np.max(p_scores_ratio) > best_score_ratio:
            best_score_ratio = np.max(p_scores_ratio)
            best_bpm_ratio = bpm_val
            
        # --- OPTION C: Pure Hit Density (The Simplest Approach) ---
        p_scores_density = np.sum(weighted_buffer[None, :] * template_peaks, axis=1)
        
        if np.max(p_scores_density) > best_score_density:
            best_score_density = np.max(p_scores_density)
            best_bpm_density = bpm_val

    return best_bpm_pearson, best_bpm_ratio, best_bpm_density


In [ ]:
# THE INITIAL SWEEP (Used once at startup to find the first class)
def localized_continuous_phase_sweep(odf_buffer, center_bpm, search_radius=1.5, step=0.3, expected_phase=None, tau_power=1.0):
    odf_size = len(odf_buffer)
    decay_curve = np.exp(-1.5 * np.linspace(1.0, 0.0, odf_size))
    weighted_buffer = odf_buffer * decay_curve
    
    # Pre-compute zero-mean buffer for Pearson
    buffer_mean = np.mean(weighted_buffer)
    buffer_centered = weighted_buffer - buffer_mean
    buffer_std = np.sqrt(np.sum(buffer_centered**2)) + 1e-6
    
    best_overall_score = -float('inf')
    best_overall_bpm = center_bpm
    best_overall_p = 0
    
    buffer_indices = np.arange(odf_size)
    bpm_evals = np.arange(max(50.0, center_bpm - search_radius), min(220.0, center_bpm + search_radius + step/2), step)
    
    btrack_fps = 60.0
    const_part = buffer_indices - (odf_size - 1)
    
    for bpm_val in bpm_evals:
        tau_val = 60.0 * btrack_fps / bpm_val
        p_max = int(np.ceil(tau_val))
        
        p_arr = np.arange(p_max)[:, None]
        phase_float = (const_part[None, :] + p_arr) % tau_val
        norm_phi = phase_float / tau_val 
        
        # Sharp triangle pulse
        beat_dist = np.minimum(norm_phi, 1.0 - norm_phi)
        template_vals = np.full((p_max, odf_size), -0.5)
        mask_beat = beat_dist < 0.1
        template_vals[mask_beat] = 1.0 - (beat_dist[mask_beat] / 0.1)
        
        template_mean = np.mean(template_vals, axis=1, keepdims=True)
        template_centered = template_vals - template_mean
        template_std = np.sqrt(np.sum(template_centered**2, axis=1)) + 1e-6
        
        # True Pearson Correlation! No more tau_power or expected_beats!
        p_scores = np.sum(buffer_centered[None, :] * template_centered, axis=1) / (buffer_std * template_std)
        
        if expected_phase is not None:
            expected_p = (expected_phase * tau_val) % tau_val
            dist_p = np.minimum(np.abs(p_arr[:, 0] - expected_p), tau_val - np.abs(p_arr[:, 0] - expected_p))
            norm_dist = dist_p / tau_val
            phase_inertia = np.exp(-0.5 * (norm_dist / 0.35)**2)
            p_scores = p_scores * (0.5 + 0.5 * phase_inertia)
            
        tau_max_score = np.max(p_scores)
        best_p = np.argmax(p_scores)
        
        gaussian_weight = np.exp(-0.5 * ((bpm_val - center_bpm) / (search_radius * 1.5))**2)
        weighted_score = tau_max_score * (0.8 + 0.2 * gaussian_weight)
        
        if weighted_score > best_overall_score:
            best_overall_score = weighted_score
            best_overall_bpm = bpm_val
            best_overall_p = best_p
            
    optimal_tau = 60.0 * btrack_fps / best_overall_bpm
    precise_phase = best_overall_p / optimal_tau
            
    return best_overall_bpm, precise_phase, best_overall_score


In [ ]:
def run_simulation(y_list):
    print(f"\n🚀 Starting SIMPLIFIED Harmonic Math Simulation...")
    infos = default_infos()
    infos["printAsservmentDetails"] = False 
    infos["useMicrophone"] = True

    SIMULATED_FPS = 60.0
    TIME_PER_FRAME = 1.0 / SIMULATED_FPS
    CHUNK_SIZE_FOR_60FPS = int(44100 / SIMULATED_FPS)

    class MockTime:
        def __init__(self): self.current_time = 0.0
        def time(self): return self.current_time

    mock_timer = MockTime()
    ListenerModule.time.time = mock_timer.time

    listener = ListenerModule.Listener(infos)
    listener.ingestion.momentum_multiplier = 0.01
    listener.dynamic_audio_latency = 0

    y_full = np.concatenate(y_list)
    mic = Robust_Simulated_Microphone(y_full, listener.ingestion.fft_band_values, infos)
    listener.hasBeenSilenceCalibrated = True
    listener.hasBeenBBCalibrated = True
    listener.ingestion.calibrate_silence = lambda fps_ratio: None
    listener.ingestion.calibrate_bb = lambda fps_ratio: None

    # Metrics
    history_time = []
    history_raw_bpm = []
    history_pearson = []
    history_ratio = []
    history_density = []
    history_class = []
    history_ltm_class = []

    standalone_phase = 0.0
    standalone_beat_count = 0
    last_standalone_beat_count = 0

    audio_time = 0.0
    playhead_time = 0.0
    frame = 0

    frames_since_sweep = 0
    frames_between_sweep = int(5 * SIMULATED_FPS)

    long_term_class = 0.0

    while mic.pop_chunk(CHUNK_SIZE_FOR_60FPS):
        mic.calculate_fft()
        listener.update()
        
        if frames_since_sweep >= frames_between_sweep:
            total_latency = listener.dynamic_audio_latency + listener.analyzer.hardware_latency
            
            # 1. RAW SWEEP: Run the wide search to see what the music is doing (it might ping-pong!)
            raw_bpm, raw_phase, raw_score = localized_continuous_phase_sweep(
                listener.analyzer.odf_buffer, center_bpm=120.0, search_radius=40.0, step=0.5, expected_phase=None)
            
            # 2. CLASS MATH: Convert raw BPM to our logarithmic circle
            current_class = bpm_to_class(raw_bpm)
            
            # 3. ALIGN & SMOOTH: Find the shortest distance on the circle, neutralizing any ping-ponging!
            min_d, aligned_class = harmonic_alignment(current_class, long_term_class)
            
            if playhead_time < 5.0:
                long_term_class = aligned_class
            else:
                # Smooth the class (if it's an octave jump, min_d is 0, diff is 0, it stays perfectly stable!)
                diff = (aligned_class - long_term_class + 0.5) % 1.0 - 0.5
                long_term_class = (long_term_class + 0.1 * diff) % 1.0  # Smooth gliding
                
            # 4. CANDIDATES: Generate exact harmonic multiples from our smoothed, stable class
            candidates = class_to_bpm_candidates(long_term_class)
            
            # 5. FINAL LOCK: Evaluate ONLY those candidates using the 3 options
            bpm_pearson, bpm_ratio, bpm_density = evaluate_specific_bpms(
                listener.analyzer.odf_buffer, candidates, expected_phase=None, tau_power=1.0)

            listener.analyzer.bpm = bpm_density
            
            frames_since_sweep = 0
            
        frames_since_sweep += 1
        
        if frame % int(SIMULATED_FPS) == 0:
            history_time.append(playhead_time)
            history_raw_bpm.append(raw_bpm if 'raw_bpm' in locals() else 120)
            history_pearson.append(bpm_pearson if 'bpm_pearson' in locals() else 120)
            history_ratio.append(bpm_ratio if 'bpm_ratio' in locals() else 120)
            history_density.append(bpm_density if 'bpm_density' in locals() else 120)
            history_class.append(current_class if 'current_class' in locals() else 0)
            history_ltm_class.append(long_term_class)

        audio_time += TIME_PER_FRAME
        mock_timer.current_time += TIME_PER_FRAME
        playhead_time += TIME_PER_FRAME
        frame += 1

    ListenerModule.time.time = time.time 
    
    print("Simulation Complete!")
    return history_time, history_raw_bpm, history_pearson, history_ratio, history_density, history_class, history_ltm_class



In [ ]:
import matplotlib.pyplot as plt

h_time, h_raw_bpm, h_pearson, h_ratio, h_density, h_class, h_ltm_class = run_simulation(y_list)

# Calculate song transition times
song_durations = [len(y)/44100.0 for y in y_list]
song_start_times = [0.0]
for d in song_durations[:-1]:
    song_start_times.append(song_start_times[-1] + d)

song_names = ['Palladium (145)', 'Pumped Up Kicks (128)', 'Nobody Rules the Streets (128)']
end_time = song_start_times[-1] + song_durations[-1]

plt.figure(figsize=(15, 18))

def draw_song_lines(y_pos=235):
    for i, start_time in enumerate(song_start_times):
        plt.axvline(start_time, color='white', linestyle=':', alpha=0.7)
        plt.text(start_time + 2, y_pos, f"Song {i+1}: {song_names[i]}", color='white', fontsize=10, fontweight='bold', bbox=dict(facecolor='black', alpha=0.5))

# PLOT 1: Option A (Pearson)
plt.subplot(4, 1, 1)
plt.plot(h_time, h_raw_bpm, label='Raw Swept BPM', color='gray', alpha=0.4)
plt.plot(h_time, h_pearson, label='Option A: Pearson Correlation', color='cyan', linewidth=2)
plt.plot([song_start_times[0], song_start_times[1]], [145, 145], color='magenta', linestyle='--', label='Target 145')
plt.plot([song_start_times[1], end_time], [128, 128], color='yellow', linestyle='--', label='Target 128')
draw_song_lines()
plt.title('Option A: Pearson Correlation (Geometric Energy Norm)')
plt.ylim(40, 250)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# PLOT 2: Option B (Ratio)
plt.subplot(4, 1, 2)
plt.plot(h_time, h_raw_bpm, label='Raw Swept BPM', color='gray', alpha=0.4)
plt.plot(h_time, h_ratio, label='Option B: Peak vs Valley Ratio', color='orange', linewidth=2)
plt.plot([song_start_times[0], song_start_times[1]], [145, 145], color='magenta', linestyle='--', label='Target 145')
plt.plot([song_start_times[1], end_time], [128, 128], color='yellow', linestyle='--', label='Target 128')
draw_song_lines()
plt.title('Option B: Peak vs Valley Ratio (Sub-Harmonic Killer)')
plt.ylim(40, 250)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# PLOT 3: Option C (Density)
plt.subplot(4, 1, 3)
plt.plot(h_time, h_raw_bpm, label='Raw Swept BPM', color='gray', alpha=0.4)
plt.plot(h_time, h_density, label='Option C: Pure Hit Density', color='magenta', linewidth=2)
plt.plot([song_start_times[0], song_start_times[1]], [145, 145], color='magenta', linestyle='--', label='Target 145')
plt.plot([song_start_times[1], end_time], [128, 128], color='yellow', linestyle='--', label='Target 128')
draw_song_lines()
plt.title('Option C: Pure Hit Density (No normalization)')
plt.ylim(40, 250)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# PLOT 4: LBT Class Tracking
plt.subplot(4, 1, 4)
plt.plot(h_time, h_class, label='Raw Class (f)', color='red', alpha=0.3)
plt.plot(h_time, h_ltm_class, label='Long Term Class (Smoothed)', color='lime', linewidth=3)
draw_song_lines(y_pos=0.9)
plt.title('Logarithmic Base Tempo (LBT) Class Tracking')
plt.ylim(-0.1, 1.1)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

